# Image Captioning: RNN/LSTM (Modular)
Notebook modular untuk preprocessing, training, dan evaluasi captioning.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

sys.path.insert(0, str(Path('src/rnn-lstm').resolve()))
sys.path.insert(0, str(Path('src/utils').resolve()))

import preprocess_flickr8k
import train_decoder
import evaluate_captioning
from caption_model import CaptionModel
from feature_extraction import load_features
from text_utils import load_vocab, load_flickr8k_captions

In [ ]:
# Sesuaikan path ini sesuai environment
captions_file = Path('/path/to/captions.txt')
train_split = Path('/path/to/Flickr_8k.trainImages.txt')
val_split = Path('/path/to/Flickr_8k.devImages.txt')
test_split = Path('/path/to/Flickr_8k.testImages.txt')

splits_json = Path('outputs/rnn_lstm/preprocess/splits.json')
vocab_path = Path('outputs/rnn_lstm/preprocess/vocab.json')
features_path = Path('/path/to/flickr8k_features.npy')
model_path = Path('/path/to/model.keras')

output_dir = Path('outputs/rnn_lstm/notebook')
output_dir.mkdir(parents=True, exist_ok=True)

## Preprocess captions

In [ ]:
if captions_file.exists() and train_split.exists() and val_split.exists() and test_split.exists():
    summary = preprocess_flickr8k.preprocess(
        captions_file=captions_file,
        train_split=train_split,
        val_split=val_split,
        test_split=test_split,
        output_dir=output_dir / 'preprocess',
        min_freq=1,
    )
    summary
else:
    print('Set path dataset dulu.')

## Train (single config)

In [ ]:
import json

if captions_file.exists() and splits_json.exists() and vocab_path.exists() and features_path.exists():
    captions = load_flickr8k_captions(str(captions_file))
    with splits_json.open("r", encoding="utf-8") as handle:
        splits = json.load(handle)
    word2idx, _ = load_vocab(vocab_path)
    features = load_features(str(features_path))
    config = train_decoder.DecoderExperimentConfig(
        decoder_type="lstm",
        n_layers=1,
        hidden_size=256,
        embed_dim=256,
        max_seq_len=35,
        feature_dim=2048,
        learning_rate=1e-3,
        injection_method="pre",
    )
    train_decoder.train_one(
        config=config,
        captions=captions,
        splits={"train": splits["train"], "val": splits["val"], "test": splits["test"]},
        features=features,
        word2idx=word2idx,
        output_root=output_dir / "train",
        batch_size=32,
        epochs=1,
        seed=42,
    )
else:
    print("Set path preprocess dan features dulu.")

## Evaluasi BLEU-4 dan METEOR

In [ ]:
import json

if model_path.exists() and splits_json.exists() and vocab_path.exists() and features_path.exists() and captions_file.exists():
    captions = load_flickr8k_captions(str(captions_file))
    with splits_json.open("r", encoding="utf-8") as handle:
        splits = json.load(handle)
    word2idx, idx2word = load_vocab(vocab_path)
    features = load_features(str(features_path))
    image_names = [name for name in splits["test"] if name in captions and name in features]
    keras_model = tf.keras.models.load_model(model_path)
    result = evaluate_captioning.evaluate(
        model=keras_model,
        captions=captions,
        image_names=image_names,
        features=features,
        word2idx=word2idx,
        idx2word=idx2word,
        decoder_type="lstm",
        max_len=30,
        mode="keras",
        seed=42,
        decoding="greedy",
        beam_size=3,
        length_penalty=0.7,
    )
    result["metrics"]
else:
    print("Set path preprocess, features, dan model dulu.")